# Enriching Archival Metadata for People Discovery

This Jupyter Notebook will allow you to follow the lesson on the drive rather than locally.
For the purpose of the workshop, we will demonstrate how to extract text data from PDFs for Named Entity Recognition.

### Installing required libraries and models

You will need to make sure that the following libraries are installed in the Google Colab. Some of these might be installed already, but it can't hurt to install them again.

In [4]:
#installing libraries and models needed for workshop
%pip install pdfplumber
%pip install pandas
%pip install spacy
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### 1. Extracting Text with pdfplumber

In [7]:
#importing libraries needed for this section
import pdfplumber
import pandas as pd
from pathlib import Path
import zipfile

In [13]:
#using zipfile to access our data
with zipfile.ZipFile("/content/enablar_lesson_6.zip", "r") as zip_file:
  zip_file.extractall("AA_Weekly_data")

In [26]:
#function to get all the pdf files in a givendirectory
def get_pdfs(dir): #dir refers to directory
   files = []
   #the asterisk * below is a wildcard, meaning what comes before the file extension does not matter
   for path in Path(dir).glob("*.pdf"):
       files.append(path)
   return files

In [27]:
#get all files from our unzipped directory
aa_files = get_pdfs("AA_Weekly_data")

In [29]:
#function to extract text from pdf files
def pdf_to_df(files):
   rows = []
   for doc in files:
       with pdfplumber.open(doc) as pdf:
           full_text = []
           for page in pdf.pages:
               text = page.extract_text()
               if text:
                   full_text.append(text)
           combined_pdf = " ".join(full_text)
       #append to rows and use Path to only keep the file name
       rows.append({"file_name": Path(doc).name, "text": combined_pdf})
       #turn into a DataFrame
   df = pd.DataFrame(rows)
   return df


In [30]:
#apply the function and check it worked by printing
aa_files_data = pdf_to_df(aa_files)
print(aa_files_data['text'])

0     AA Weekly http://www.aaschool.ac.uk/PUBLIC/WHA...
1     AA Weekly http://www.aaschool.ac.uk/PUBLIC/WHA...
2     View this email in your browser\nMonday 9 – Sa...
3     View this email in your browser\nMonday 30 – S...
4     AA Weekly http://www.aaschool.ac.uk/PUBLIC/WHA...
5     AA Weekly http://www.aaschool.ac.uk/PUBLIC/WHA...
6     AA Weekly http://www.aaschool.ac.uk/PUBLIC/WHA...
7     AA Weekly http://www.aaschool.ac.uk/PUBLIC/WHA...
8     View this email in your browser\nMonday 7 – Sa...
9     19/03/2018 AA Weekly\nView this email in your ...
10    http://www.aaschool.ac.uk/PUBLIC/WHATSON/aawee...
11    View this email in your browser\nMonday 14 – S...
12    View this email in your browser\nMonday 23 – S...
13    AA Weekly http://www.aaschool.ac.uk/PUBLIC/WHA...
14    View this email in your browser\nMonday 21 – S...
15    AA Weekly http://www.aaschool.ac.uk/PUBLIC/WHA...
16    19/03/2018 AA Weekly\nView this email in your ...
Name: text, dtype: object


### 2. Running the NER

In [31]:
import spacy
#load spacy moodel
nlp = spacy.load("en_core_web_lg")


In [43]:
#ner function
def run_ner(text, nlp): #arguments are text and the nlp model
   entities = []

  #creating a doc object
   doc = nlp(text)
   for ent in doc.ents:
       if ent.label_ == "PERSON": #keeping only PERSON entities and appending it to our entities list
           entities.append(
           ent.text.strip()
           )
           #.strip() gets rid of leading or trailing whitespace

   return entities

In [44]:
#apply NER; make sure to specify the nlp model
aa_files_data["entities"] = aa_files_data["text"].apply(lambda x: run_ner(x, nlp))


In [45]:
#explode df
aa_data_exploded = aa_files_data.explode("entities")

In [46]:
#group by ner column; the starting and ending parentheses are just a way for Python to treat this as one
#continous line of code despite the line breaks
aa_data_deduped = (
   aa_data_exploded.groupby("entities", as_index=False)
   .agg(source_files=("file_name", lambda x: "; ".join(sorted(x.unique()))))
)

In [47]:
#print off the dataframe to verify if it needs cleaning (spoiler: it does)
aa_data_deduped

,entities,source_files
0,AArchitecture,Weekly_201718_Term 2_Week 4.pdf; Weekly_201718...
1,Aalto,Weekly_201718_Term 2_Week 11.pdf
2,Ada Louise Huxtable Prize,Weekly_201718_Term 2_Week 6.pdf
3,Adam,Weekly_201718_Term 3_Week 12.pdf
4,Adam Olivia Sudjic,Weekly_201718_Term 3_Week 12.pdf
...,...,...
252,Zachary Mollica,Weekly_201718_Term 3_Week 2.pdf
253,Zsuzsa,Weekly_201718_Term 3_Week 12.pdf
254,Zsuzsa Peter,Weekly_201718_Term 3_Week 12.pdf
255,aaschool.ac.uk/whatson,Weekly_201718_Term 2_Week 5_Open_Week.pdf


In [39]:
#data cleaning

#keep only rows with multiple token names (if there is no space = one word)
aa_data_deduped = aa_data_deduped[aa_data_deduped["entities"].str.contains(" ")]


#use regex and a mask “~” to get rid of rows containing any digits
aa_data_deduped = aa_data_deduped[~aa_data_deduped["entities"].str.contains(r"\d")]

# use a mask “~” to get rid of rows containing "AA"
aa_data_deduped = aa_data_deduped[~aa_data_deduped["entities"].str.contains("AA")]
